# Generative Models to Estimate the Gradients of the Data Distribution

Source: <BR>
https://github.com/JeongJiHeon/ScoreDiffusionModel/tree/main/NCSN

Adapted:

Antonio Esteves @UMinho, February 2025

---

## Model Definition

First, we defined the perturbed data distribution.

$$
q_{\sigma}(x)=\int p \mathcal(x \vert t, \sigma^2\mathbf{I}) d\sigma dt \tag{1}
$$

However, $s_{\theta}(x) = \nabla_{x} \log q_{\sigma}(x) \approx \nabla_{x} \log p(x)$ is true only when the noise is small enough, such that, $q_{\sigma}(x) \approx p(x)$.
By adding noise with multiple variances, we obtain a sequence of noise-perturbed distributions that converge to the true data distribution.
<br>

The conditional perturbed data distribution for step $i$ is: <BR>

$$
q_{\sigma_{i}}(\tilde{x} \vert x)=\mathcal{N}(\tilde{x} ; x, \sigma_{i}^2\mathbf{I}) \tag{2}
$$

This distribution can be reparameterized as: <BR>

$$
\tilde{x}=x+\sigma_{i}*z \tag{3}
$$

where<BR>

$$
z \sim \mathcal{N}(0, \mathbf{I}) \tag{4}
$$

and in each step the noise variance is bigger than in the next step:<BR>

$$
\frac{\sigma_{1}}{\sigma_{2}}=\frac{\sigma_{2}}{\sigma_{3}}=\cdots=\frac{\sigma_{L-1}}{\sigma_{L}} > 1 \tag{5}
$$

Next, we define a score model that depends on the variances.

$$
\mathbf{s}_{\theta}(\tilde{x}, \sigma_i) \approx \nabla_{\tilde{x}} \log{q_{\sigma_i} }(\tilde{x}\vert{x}) \tag{6}
$$


Using the expression of a Gaussian with mean $x$ and variance $\sigma_i^2$, the score of perturbed data distribution is:

$$
\begin{align}
\begin{aligned}
\nabla_{\tilde{x}}\log q_{\sigma_i}(\tilde{x}\vert{x}) & = 
\frac{d}{d\tilde{x}} \log \left( constant*e^{-\frac{(\tilde{x}-x)^2}{2\sigma_i^2}} \right) \\
& = - \frac{1}{2}\frac{d}{d\tilde{x}} \left(\frac{(\tilde{x}-x)^2}{\sigma_i^2} \right) \\
& = - \frac{2}{2} \frac{\tilde{x}-x}{\sigma_i} \frac{d}{d\tilde{x}} \frac{\tilde{x}-x}{\sigma_i} \\
& = - \frac{(\tilde{x}-x)}{\sigma_i^2} \\
& = -\frac{z}{\sigma_i} \text{(using equation 3, } z = \frac{\tilde{x}-x}{\sigma_i} \text{)}
\end{aligned}
\tag{7}
\end{align}
$$ 

Then, we can use the objective function for noise conditional score network via score matching.

$$
\mathcal{L}(\theta, \{ \sigma_{i}\}_{i=1}^L) = \frac{1}{L}\sum_{i=1}^L\lambda({\sigma_i})\mathcal{l}(\theta;\sigma_i) \tag{8}
$$

where

$$\mathcal{l}(\theta;\sigma_i) = \frac{1}{2}\mathbb{E}_{x \sim p(x)} \mathbb{E}_{\tilde{x}\sim\mathcal{N}(x,\sigma_i^2\mathbf{I})} \left[ \Vert \mathbf{s}_{\theta}(\tilde{x}, \sigma_i)+\frac{\tilde{x}-x}{\sigma_i^2}\Vert^2_2 \right] \tag{9}
$$

and we will opt for $\lambda$ given by

$$\lambda(\sigma_i) = \sigma_i^2 \tag{10}$$

## The score model

In [ ]:
import math
import torch
import torch.nn                    as     nn
import matplotlib.pyplot           as     plt
import matplotlib.animation        as     animation
import numpy                       as     np
from   IPython.display             import HTML
from   IPython.display             import clear_output

In [ ]:
class ScoreModel(nn.Module):

    def __init__(self, device, L_steps, sigma_min, sigma_max, p=0.5):
        '''
        Score model network.

        L_steps   : number of perturbation schedule steps (Langevin dynamics steps).
        sigma_min : minimum sigma in the perturbation schedule.
        sigma_min : maximum sigma in the perturbation schedule.
        p         : dropout probability.        
        '''
        super().__init__()
        self.device = device
        # Define the noise variance to apply in each Langevin dynamics step
        self.sigmas = torch.exp(
            torch.linspace(
                start = math.log(sigma_max),
                end   = math.log(sigma_min),
                steps = L_steps,
            )
        ).to(device = device)

        self.linear_model1 = nn.Sequential(
            nn.Linear(2, 256),
            nn.Dropout(p),
            nn.GELU(),
        )

        # Embedding layer to encode the variances (sigmas), 
        # which will be used for conditioning the score model. 
        self.embedding_layer = nn.Embedding(L_steps, 256)

        self.linear_model2 = nn.Sequential(
            nn.Linear(256, 512),
            nn.Dropout(p),
            nn.GELU(),

            nn.Linear(512, 512),
            nn.Dropout(p),
            nn.GELU(),

            nn.Linear(512, 2),
        )

        self.to(device = self.device)


    def loss_fn(self, x, idx=None):
        '''
        The loss function is used only in the training phase.
        It performs the forward computations of the model.

        x   : Use real data if idx==None, else use perturbed data.
        idx : 'idx' must be set to 'None' during training; then, during the
              forward computation it is initialized  'idx' with random ids
              between 0 and L-1. These ids will be used to selected the of
              the noise that will perturb the data samples. 
              for inference, 'idx' must be defined with a value (not 'None').
              It is recommended that you specify 'idx'.
         '''
        scores, target, sigma = self.forward(x, idx=idx, get_target=True)
        target = target.view(target.shape[0], -1)
        scores = scores.view(scores.shape[0], -1)
        
        # loss = mean [(s_theta - noise/sigma)^2]*sigma^2
        losses = torch.square(scores - target).mean(dim=-1) * sigma.squeeze() ** 2

        return losses.mean(dim=0)


    def forward(self, x, idx=None, get_target=False):
        '''
        x          : 'x' is real data if 'idx=None', else 'x' is the perturbed data.
        idx        : 'idx' must be set to 'None' during training; then, during the
                     forward computation it is initialized  'idx' with random ids
                     between 0 and L-1. These ids will be used to selected the of
                     the noise that will perturb the data samples. 
                     for inference, 'idx' must be defined with a value (not 'None').
                     It is recommended that you specify 'idx'.
        get_target : if 'True' (training phase), the method returns 'target' and 'sigma', 
                     besides 'output' (score prediction)
        '''

        # Training phase .......................................
        if idx == None:
            idx = torch.randint(
                0,
                len(self.sigmas),
                (x.size(0), ),
            ).to(device = self.device)
            used_sigmas = self.sigmas[idx][:,None]
            noise       = torch.randn_like(x)
            x_tilde     = x + noise * used_sigmas

        # Sampling phase .....................................
        else:
            idx = torch.cat(
                [torch.Tensor([idx for _ in range(x.size(0))])]
            ).long().to(device = self.device)
            used_sigmas = self.sigmas[idx][:,None]
            x_tilde     = x

        if get_target:
            target = - noise / used_sigmas

        output    = self.linear_model1(x_tilde)
        embedding = self.embedding_layer(idx)
        output    = self.linear_model2(output + embedding)

        # output      = s_theta  = scores
        # target      = -z/sigma = (x_tilde - x)/sigma^2
        # used_sigmas = sigma

        return (output, target, used_sigmas) if get_target else output

## Sampling with annealed Langevin dynamics

$$
\tilde{x}_t = \tilde{x}_{t-1} + \frac{\alpha_{i}}{2} \nabla_{\tilde{x}_{t-1}} \log p_{\sigma_{i}}(\tilde{x}_{t-1}) + \sqrt{\alpha_{i}}z_{t}  \text{, where } i \in [1, L]  \text{ and }  t \in [1,T] \tag{11}
$$

In [ ]:
class AnnealedLangevinDynamics():
    '''
    Class to sample a score model using annealed Langevin dynamics.
    '''
    def __init__(self, sigma_min, sigma_max, L_steps, T_steps, score_fn, device, epsilon = 1e-1):
        '''
        sigma_min : minimum variance of perturbation schedule
        sigma_max : maximum variance of perturbation schedule
        L         : iteration step of Langevin dynamics
        T         : annealed step of annealed Langevin dynamics
        score_fn  : trained score network
        epsilon   : coefficient of step size
        '''
        # Create the L variance values, regularly spaced, between the 
        # selected maximum and minimum.
        self.sigma = torch.exp(
            torch.linspace(
                start = math.log(sigma_max),
                end   = math.log(sigma_min),
                steps = L_steps,
            )
        )
        # alpha_i = epsilon * (sigma_i/sigma_L)^2
        self.alpha          = epsilon * (self.sigma / self.sigma[-1] ) ** 2
        self.score_fn       = score_fn
        self.annealed_steps = T_steps
        self.device         = device

    def _one_annealed_step_iteration(self, x, idx):
        '''
        x   : perturbed data
        idx : step of perturbation schedule
        '''
        self.score_fn.eval()
        # Draw (sampling_number x 2) values from N(0,1)
        z         = torch.randn_like(x).to(device = self.device)
        alpha     = self.alpha[idx]

        # Compute next value for samples x given current sample values and current alpha value
        x         = x + 0.5 * alpha * self.score_fn(x, idx) + torch.sqrt(alpha) * z
        return x

    def _one_annealed_step(self, x, idx):
        '''
        x   : perturbed data
        idx : step of perturbation schedule
        '''
        # For one noise level, run T annealing steps
        for _ in range(self.annealed_steps):
            x = self._one_annealed_step_iteration(x, idx)
        return x

    def _one_perturbation_step(self, x):
        '''
        x   : sample frthe prior distribution U[0,1).
        '''
        # Iterate over the L noise variance levels
        for idx in range(len(self.sigma)):
            # Execute one annealed LD step
            x = self._one_annealed_step(x, idx)
            # Make 'x' an output of the iterative process
            yield x

    @torch.no_grad()
    def sampling(self, sampling_number, only_final=False):
        '''
        Draw samples from the score model using annealed Langevin dynamics.
        only_final : If True, the method returns only the output of the final schedule step.
        '''
        # Get sampling_number pairs from the prior, which is a Uniform distribution 
        # in the interval [0,1) and convert the values to the range [-1,+1].
        sample        = (torch.rand([sampling_number,2]).to(device = self.device) - 0.5)*2
        sampling_list = []

        final = None
        for sample in self._one_perturbation_step(sample):
            final = sample
            if not only_final:
                sampling_list.append(final)

        return final if only_final else torch.stack(sampling_list)

In [ ]:
class AverageMeter(object):
    '''
    Class for keep track of a metric (the loss) during training and calculate its average.
    '''
    def __init__(self, name, fmt=':f'):
        self.name  = name
        self.fmt   = fmt
        self.reset()

    def reset(self):
        self.val   = 0
        self.avg   = 0
        self.sum   = 0
        self.count = 0

    def update(self, val, n=1):
        self.val     = val
        self.sum    += val * n
        self. count += n
        self.avg     = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)


class ProgressMeter(object):
    '''
    Helper class to print the batch and total batches well formated during training.
    '''
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters       = meters
        self.prefix       = prefix

    def display(self, batch):
        entries  = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]

        print('\r' + '\t'.join(entries), end = '')

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

In [ ]:
def scatter(sample, only_final, scatter_range = [-10, 10]):
    clear_output()
    if only_final:
        scatter = sample.detach().cpu().numpy()
        scatter_x, scatter_y = scatter[:,0], scatter[:,1]
        plt.figure(figsize=(7, 7))

        plt.xlim(scatter_range)
        plt.ylim(scatter_range)
        plt.rc('axes', unicode_minus=False)

        plt.scatter(scatter_x, scatter_y, s=5)
        plt.show()

    else:
        step_size = sample.size(0)
        fig, axs  = plt.subplots(
            1,
            step_size,
            figsize            = (step_size * 4, 4),
            constrained_layout = True,
        )
        for i in range(step_size):
            scatter              = sample[i].detach().cpu().numpy()
            scatter_x, scatter_y = scatter[:,0], scatter[:,1]
            axs[i].scatter(scatter_x, scatter_y, s=5)
            axs[i].set_xlim(scatter_range)
            axs[i].set_ylim(scatter_range)
        plt.show()

# Example 1

A distribution that is a mixture of two bi-variate Gaussian, one centered in (3,3) and variance 1 and the other centered in (-3,-3) and variance 1. The first Gaussian is sampled 20% of the time and the second one 80% of the time.


$$p(x) = \frac{1}{5} \times \mathcal{N}((3,3), I) + \frac{4}{5} \times \mathcal{N}((-3,-3), I) \tag{12}$$

In [ ]:
class DataSet(torch.utils.data.Dataset):
    '''
    Dataset that provides samples from a distribution that is a mixture of two Gaussian.
    '''
    def __init__(self, dist1, dist2, shape = (2), probability=0.2, total_len = 1000000):
        self.dist1_mean, self.dist1_var = dist1[0], dist1[1]
        self.dist2_mean, self.dist2_var = dist2[0], dist2[1]
        self.shape       = shape
        self.probability = probability
        self.total_len   = total_len

    @property
    def get_probability(self):
        # Returns a random boolean value that is 'True' with probability=0.2 and 
        # is 'False' with probability=1-0.2=0.8
        return torch.rand(1) < self.probability

    @property
    def _sampling_1(self):
        return self.dist1_mean + torch.randn(self.shape) * self.dist1_var

    @property
    def _sampling_2(self):
        return self.dist2_mean + torch.randn(self.shape) * self.dist2_var

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        # Returns a sample from distribution 1 with probability=0.2 and
        # and a sample from distribution 2 with probability=1-0.2=0.8
        data = self._sampling_1 if self.get_probability else self._sampling_2

        return data

## Hyperparameters of the model and perturbation process

In [ ]:
epsilon        = 1e-6
sigma_min      = 0.001
sigma_max      = 10
n_steps        = 10   # L
annealed_steps = 100  # T
device         = torch.device('cuda')

In [ ]:
model    = ScoreModel(device, n_steps, sigma_min, sigma_max, p = 0.3)
optim    = torch.optim.Adam(model.parameters(), lr = 0.005)
dynamics = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)

## Hyperparameters of the training process

In [ ]:
scatter_range     = [-10, 10]
total_iterations  = 1000
current_iteration = 0
display_iteration = 200
sampling_number   = 1000
only_final        = True
losses            = AverageMeter('Loss', ':.4f')
progress          = ProgressMeter(total_iterations, [losses], prefix='Iteration ')

In [ ]:
dist1        = (3, 1)  # mean : (3, 3),   std : 1
dist2        = (-3, 1) # mean : (-3, -3), std : 1
probability  = 0.2
batch_size   = 8192 * 2
dataloader   = torch.utils.data.DataLoader(
    DataSet(
        dist1,
        dist2,
        probability = probability,
        total_len   = batch_size * total_iterations,
    ),
    batch_size = batch_size,
    drop_last  = True,
)
dataiterator = iter(dataloader)

In [ ]:
scatter(next(iter(dataloader)), True)

## Training the Score Model

In [ ]:
while current_iteration != total_iterations:
    try:
        data = next(dataiterator)
    except:
        dataiterator = iter(dataloader)
        data = next(dataiterator)
    data = data.to(device = device)
    loss = model.loss_fn(data)

    optim.zero_grad()
    loss.backward()
    optim.step()

    losses.update(loss.item())
    progress.display(current_iteration)

    current_iteration += 1

    # Perform sampling every 'display_iteration' iterations
    if current_iteration % display_iteration == 0:
        dynamics = AnnealedLangevinDynamics(
            sigma_min,
            sigma_max,
            n_steps,
            annealed_steps,
            model,
            device,
            epsilon = epsilon,
        )
        sample  = dynamics.sampling(sampling_number, only_final)
        scatter(sample, only_final, scatter_range = scatter_range)
        losses.reset()


## Sampling the Score Model

In [ ]:
sampling_number = 1000
only_final      = True
dynamics        = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)
sample     = dynamics.sampling(sampling_number, only_final)
scatter(sample, only_final, scatter_range= scatter_range)

In [ ]:
sampling_number = 1000
only_final      = False
dynamics        = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)
sample     = dynamics.sampling(sampling_number, only_final)
scatter(sample, only_final, scatter_range=scatter_range)

In [ ]:
def update_plot(i, data, scat):
    scat.set_offsets(data[i].detach().cpu().numpy())
    return scat

numframes            = len(sample)
scatter_point        = sample[0].detach().cpu().numpy()
scatter_x, scatter_y = scatter_point[:,0], scatter_point[:,1]

fig = plt.figure(figsize=(6, 6))
plt.xlim(scatter_range)
plt.ylim(scatter_range)
scat = plt.scatter(scatter_x, scatter_y, s=1)
plt.show()
clear_output()

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames   = range(numframes),
    fargs    = (sample, scat),
    interval = 150,
)
# ani.save('ncsn_example1.gif')
HTML(ani.to_jshtml())

# Example 2

In [ ]:
class DataSet2(torch.utils.data.Dataset):

    def __init__(self, dist, shape = (2), total_len = 1000000):
        self.dist_mean, self.dist_var, self.dist_num = dist[0], dist[1], dist[2]
        self.distribution_list = np.stack(
            [
            np.dot(
                [
                    [np.cos(angle), -np.sin(angle)],
                    [np.sin(angle), np.cos(angle)]
                ], 
                dist[0]
            ) for angle in np.linspace(start=0, stop=2 * np.pi, num = dist[2]+1)
            ]
        )
        self.shape             = shape
        self.total_len         = total_len

    @property
    def _choose_distribution(self):
        return int(torch.rand(1) * self.dist_num)

    @property
    def _sampling(self):
        dist_idx = self._choose_distribution
        return torch.from_numpy(self.distribution_list[dist_idx]).float() + torch.randn(self.shape) * self.dist_var

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        data = self._sampling

        return data

$$p(x) = \sum_{i=1}^{N}\frac{1}{N}\mathcal{N}(\mu *(\cos{\frac{2\pi i}{N}},\sin{\frac{2\pi i}{N}}), \sigma^2) \tag{13}$$

## Hyperparameters of the model and perturbation process

In [ ]:
epsilon        = 1e-5
sigma_min      = 0.005
sigma_max      = 10
n_steps        = 10
annealed_steps = 100
device         = torch.device('cuda')

In [ ]:
model    = ScoreModel(device, n_steps, sigma_min, sigma_max, p = 0.3)
optim    = torch.optim.Adam(model.parameters(), lr = 0.005)
dynamics = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)

## Hyperparameters of the Training Process

In [ ]:
total_iteration   = 2000
current_iteration = 0
display_iteration = 200
scatter_range     = [-20, 20]
sampling_number   = 1000
only_final        = True

losses   = AverageMeter('Loss', ':.4f')
progress = ProgressMeter(total_iteration, [losses], prefix='Iteration ')

In [ ]:
dist       = [[10,0], 1, 8] # mu, sigma, N used in equation 13
batch_size = 8192 * 2
dataloader = torch.utils.data.DataLoader(
    DataSet2(
        dist,
        total_len = batch_size * total_iterations
    ),
    batch_size = batch_size,
    drop_last  = True,
)
dataiterator = iter(dataloader)

In [ ]:
scatter(next(iter(dataloader)), True, scatter_range = [-14, 14])

## Training the Score Model

In [ ]:
while current_iteration != total_iteration:
    try:
        data = next(dataiterator)
    except:
        dataiterator = iter(dataloader)
        data = next(dataiterator)
    data = data.to(device = device)
    loss = model.loss_fn(data)

    optim.zero_grad()
    loss.backward()
    optim.step()

    losses.update(loss.item())
    progress.display(current_iteration)

    current_iteration += 1

    if current_iteration % display_iteration == 0:
        dynamics = AnnealedLangevinDynamics(
            sigma_min,
            sigma_max,
            n_steps,
            annealed_steps,
            model,
            device, 
            epsilon = epsilon,
        )
        sample = dynamics.sampling(sampling_number, only_final)
        scatter(sample, only_final, scatter_range = scatter_range)


## Sampling the Score Model

In [ ]:
sampling_number = 10000
only_final = True
dynamics   = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)
sample = dynamics.sampling(sampling_number, only_final)
scatter(sample, only_final, scatter_range= scatter_range)

In [ ]:
sampling_number = 10000
only_final = False
dynamics   = AnnealedLangevinDynamics(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)
sample     = dynamics.sampling(sampling_number, only_final)
scatter(sample, only_final, scatter_range=scatter_range)

In [ ]:
def update_plot(i, data, scat):
    scat.set_offsets(data[i].detach().cpu().numpy())
    return scat

numframes     = len(sample)
scatter_point = sample[0].detach().cpu().numpy()
scatter_x     = scatter_point[:,0]
scatter_y     = scatter_point[:,1]

fig = plt.figure(figsize=(6, 6))
plt.xlim(scatter_range)
plt.ylim(scatter_range)
scat = plt.scatter(scatter_x, scatter_y, s=1)
plt.show()
clear_output()

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames   = range(numframes),
    fargs    = (sample, scat),
    interval = 150,
)
# ani.save('ncsn_example2.gif')
HTML(ani.to_jshtml())